# Module 7 — Evaluation: Making "Is This Actually Better" a Measured Answer, Not a Vibe

**The gap, from Modules 1-6:** every module so far ended with "here's what happened, one run" —
read a transcript, judge by eye whether the answer looks right. That doesn't scale past a demo,
and it can't answer the question this whole course has been building toward: is the agentic loop
(Module 2), the extra tools (Modules 3/4/6), or the dedup/resolution work (Module 5) actually
*better* than the Module 1 baseline, or does it just feel more sophisticated? Answering that
needs a **judge** — something that scores an answer against a known-correct reference, the same
way a human grader would use an answer key, run consistently across many questions and systems.

**What we build in this module:**
- An LLM-as-judge that scores an answer against a gold reference on correctness, completeness,
  and grounding — with mitigations built in for the biases an LLM judge is prone to
- A decomposed-judgment mode for multi-part questions, so one wrong sub-claim doesn't collapse
  the score for the whole answer
- A small, checked-in eval dataset of question / gold-answer pairs, drawn from vendor-verified
  data and curated narrative facts
- Aggregate metrics that turn individual judged answers into a comparable, per-system number
- Three worked examples that run the judge against this project's actual baseline and agentic
  systems, building in difficulty: a single comparison question, a cross-system comparison, and
  a multi-part question that shows why decomposition matters

**New components introduced:**
- `evaluation.validators` — `JudgeScore` (holistic) and `DecomposedJudgment` (multi-part)
  Pydantic schemas
- `evaluation.prompts` — the judge's system prompts, rubric, and debiasing instructions
- `evaluation.judge` — `judge_answer()` and `judge_answer_decomposed()`
- `evaluation.metrics` — `aggregate_scores()`, `aggregate_by_system()`, `score_decomposed()`
- `data/eval/questions.json` — a small, checked-in eval dataset (question + gold answer + how
  the gold answer was sourced)

## 1. A judge is an LLM call, with an LLM call's failure modes

Before trusting any number this notebook produces, it's worth being explicit about how a judge
can go wrong — the same way a course on RAG has to be explicit about hallucination before it can
be trusted to detect it:

| Bias | What it looks like | Mitigation in `evaluation/` |
|---|---|---|
| **Verbosity bias** | Longer, more hedged answers score higher regardless of correctness | `JUDGE_SYSTEM_PROMPT` explicitly instructs the model to ignore length/tone/confidence |
| **Halo effect** | One wrong claim drags an otherwise-mostly-right answer to "fails entirely" | `judge_answer_decomposed` — see section 4 |
| **Self-inconsistency** | A judge's itemized breakdown and its own overall number disagree | The overall score is computed in code (`metrics.score_decomposed`), never asked of the model |
| **Un-anchored drift** | Scores cluster at the "safe" middle value, or drift across questions | An explicit 1/3/5-anchored rubric embedded in the system prompt, not a bare "rate 1-5" |
| **Grounding/correctness conflation** | "Is this grounded" quietly collapses into "does this match the gold answer" | `grounding` is scored against the system's actual retrieved context when passed, not the gold answer |
| **Position/order bias** | A pairwise "which is better" judge is swayed by which answer came first | Structurally avoided — this judge only ever scores one answer against a reference, never compares two |

One more, easy to miss: **snap judgments.** `JudgeScore`'s fields are ordered
`correctness_rationale` *then* `correctness` (and so on per dimension) — since structured-output
fields are generated in declaration order, the model has to write its reasoning before it commits
to a number, not the other way around.


In [1]:
from financial_advisor.evaluation.prompts import JUDGE_SYSTEM_PROMPT

print(JUDGE_SYSTEM_PROMPT)


You are an impartial evaluator of a financial-advisor AI's answer, scoring it against a gold (reference) answer that is independently known to be correct — never score an answer in a vacuum, always relative to the gold answer provided.

Score only the answer's relationship to the question and the gold answer:
- Never let length, confidence of tone, or formatting influence a score. A short, correct answer must score as high as a long one; a long, padded answer restating the same facts must not score higher than a terse one that states them once.
- Do not reward hedging, and do not penalize appropriate uncertainty — if the gold answer itself says the source material doesn't resolve something, an answer that says the same thing is not incomplete.
- You are evaluating exactly one answer, not comparing it to any alternative — there is nothing to be ordered or ranked here.

For each dimension below, its rationale field comes first in the response schema — reason about the specific evidence b

In [2]:
from financial_advisor.evaluation.validators import JudgeScore

# Field order is the mitigation: each *_rationale precedes its score.
for name, field in JudgeScore.model_fields.items():
    print(f"{name}: {field.description}")


correctness_rationale: Compare the answer's factual content to the gold answer. Note any figures, dates, or claims that are right, wrong, or missing. Judge factual accuracy only — not style, tone, or length.
correctness: 1=contradicts the gold answer on its central claim; 3=partially correct, a mix of right and wrong or missing facts; 5=fully matches the gold answer's facts
completeness_rationale: Check the answer against every part of what the question asked, using the gold answer as the checklist. List anything the gold answer addresses that this answer omits.
completeness: 1=addresses none of what was asked; 3=addresses some but not all parts of a multi-part question; 5=addresses everything the question asked for
grounding_rationale: Check whether every claim in the answer is traceable to the retrieved context it was generated from (if provided) — or to the gold answer, when no retrieved context is given. Ignore whether an invented fact happens to be true — judge only whether the an

**One deliberate non-mitigation, worth naming:** temperature. Lower temperature would help
judge-to-judge consistency, but this project's Azure deployment rejects `temperature=0` outright
— the same constraint `text2cypher/chain.py` hit in Module 6. `judge_answer` runs at the client
default (1); the rationale-first schema and anchored rubric above are the substitute, not a
like-for-like replacement.


## 2. Gold answers: two different kinds, and why the difference matters

`judge_answer` is only as good as the `ground_truth` it's given. This project has two genuinely
different sources for one, and it's worth being honest that they carry different reliability:

1. **Structured, vendor-verified data** — `FinancialPeriod` nodes (Sharadar, via Module 3) are
   as-reported financial figures from a paid data vendor, not an LLM's read of anything. For a
   question whose answer is a number, this is as close to ground truth as this project has.
2. **Curated narrative facts** — for anything Sharadar doesn't cover (which product category
   grew fastest, what risk factor a filing names), the gold answer has to be authored — here,
   from the model's own knowledge plus web research, cross-checked against the actual filing
   content the corpus was built from. `data/eval/questions.json` records a `source` field for
   exactly this reason: a reviewer should be able to tell, for every gold answer, which kind it
   is and re-verify it independently of trusting this notebook.

The three questions below all draw on one or both. Loading the dataset first:


In [3]:
import json
from pathlib import Path

QUESTIONS = json.loads(Path("../data/eval/questions.json").read_text())
for q in QUESTIONS:
    print(f"[{q['id']}]")
    print(" Q:", q["question"])
    print(" source:", q["source"])
    print()


-net-sales-2024-yoy]
 Q: What was 3M total net sales for fiscal year 2024, and how did that compare to fiscal year 2023?
 source: Sharadar FinancialPeriod (MMM:2023-12-31, MMM:2024-12-31), cross-checked against 3M's own continuing-operations framing following the April 2024 Solventum spin-off

[apple-3m-margin-2024]
 Q: Compare Apple and 3M on fiscal year 2024 net income and total net sales: give both companies' figures and say which company had the higher net income margin.
 source: Sharadar FinancialPeriod (AAPL:2024-12-31, MMM:2024-12-31)

[apple-2024-three-part]
 Q: For fiscal year 2024, what was Apple total net sales, which product category had the fastest year-over-year growth, and what does Apple identify as a key risk related to its supply chain concentration?
 source: Sharadar FinancialPeriod (AAPL:2024-12-31) for the sales figures; Apple's FY2024 10-K (Item 1A Risk Factors, Item 7 MD&A) for the growth-driver and supply-chain content



In [4]:
from financial_advisor.services.neo4j_service import neo4j_service

# The structured half of gold-answer construction: pull the actual vendor-sourced figures
# straight from the graph, the same way agent.tools.get_financials does, instead of trusting
# the hand-typed ground_truth string in the dataset to be right.
rows = neo4j_service.run_query(
    """
    MATCH (c:Company)-[:HAS_FINANCIALS]->(fp:FinancialPeriod)
    WHERE c.id IN ["3M", "APPLE"] AND fp.calendardate IN ["2023-12-31", "2024-12-31"]
    RETURN c.id AS company, fp.calendardate AS year, fp.revenue AS revenue, fp.netinc AS netinc
    ORDER BY company, year
    """
)
for r in rows:
    print(r)


{'company': '3M', 'year': '2023-12-31', 'revenue': '32681000000', 'netinc': '-6995000000'}
{'company': '3M', 'year': '2024-12-31', 'revenue': '24575000000', 'netinc': '4173000000'}
{'company': 'APPLE', 'year': '2023-12-31', 'revenue': '383285000000', 'netinc': '96995000000'}
{'company': 'APPLE', 'year': '2024-12-31', 'revenue': '391035000000', 'netinc': '93736000000'}


These are the exact numbers `data/eval/questions.json`'s gold answers are built from —
3M's fiscal-2024 net sales of \$24,575M against a fiscal-2023 figure of \$32,681M, Apple's
\$391,035M / \$93,736M for FY2024. Worth flagging already: 3M's \$32,681M for 2023 is the
*as-reported total-company* figure — it includes the Solventum health-care business 3M spun off
in April 2024. Whether that's "the" right number to compare fiscal 2024 against, or whether the
continuing-operations restated figure is, is a real judgment call — and it comes back with a
concrete consequence in the next section.


## 3. Example 1 — a single comparison question, and a lesson about gold-answer framing

`qa.baseline.ask` (Module 1) against a straightforward year-over-year question. Retrieving the
context explicitly (rather than calling the `ask()` convenience wrapper) so it can be passed to
`judge_answer` for a real `grounding` check — not just correctness against the gold answer.


In [5]:
from financial_advisor.qa.baseline import build_context, embed_question, generate_answer, retrieve_chunks

Q1 = next(q for q in QUESTIONS if q["id"] == "3m-net-sales-2024-yoy")

chunks = retrieve_chunks(embed_question(Q1["question"]), k=5)
baseline_context = build_context(chunks)
baseline_answer = generate_answer(Q1["question"], chunks)

print("Question:", Q1["question"])
print("\nBaseline answer:\n", baseline_answer)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Question: What was 3M total net sales for fiscal year 2024, and how did that compare to fiscal year 2023?

Baseline answer:
 3M's total net sales for fiscal year 2024 were $24,575 million, compared with $24,610 million in 2023 — a decrease of $35 million, or about 0.1% year over year.


In [6]:
from financial_advisor.evaluation.judge import judge_answer

result_1 = judge_answer(Q1["question"], baseline_answer, Q1["ground_truth"], context=baseline_context)
result_1["system"] = "baseline"

print({k: v for k, v in result_1.items() if k != "rationale"})
print()
print(result_1["rationale"])


{'correctness': 3, 'completeness': 3, 'grounding': 5, 'system': 'baseline'}

Correctness: The candidate correctly states 3M's FY2024 net sales as $24,575 million and the FY2023 figure of $24,610 million, and correctly computes the decline of $35 (≈0.1%). However, the gold answer also highlights an important nuance: a total-company as-reported 2023 figure (~$32,681m) that is distorted by the April 2024 Solventum spin-off. The candidate gives only the continuing-operations comparison and omits the as-reported/spin-off context, so it partially matches the gold answer.
Completeness: The question asks for FY2024 net sales and how that compared to FY2023. The candidate provides the FY2024 amount and a direct year-over-year comparison on a continuing-operations basis (24,575 vs 24,610, -$35, ~-0.1%). The gold answer, however, also explains the distortion caused by the April 2024 Solventum spin-off and provides the as-reported 2023 figure; the candidate omits that important explanatory detail,

**What happened, in one run.** The baseline's retrieved chunks contained 3M's own
continuing-operations comparison (2024 vs. a restated ~\$24.6B 2023 figure, following the
Solventum spin-off) — a real, correct number, faithfully reported. For example, in one run the
judge scored `grounding` at 5 — every claim in the answer was traceable to what was actually
retrieved, no hallucination — but `correctness`/`completeness` came back at 3, not 5, because
the gold answer's *primary* framing leads with the as-reported \$32,681M figure, and the
baseline's answer never mentioned that number or the spin-off distortion that explains the gap.

This is a case worth naming on its own: the judge inheriting the gold answer's ambiguity. The
baseline didn't get anything *wrong* here — it answered with a different, also-defensible
framing than the one the gold answer happens to lead with. A judge, however carefully built,
cannot fix an underspecified reference; it can only faithfully apply the one it's given. That's
a real argument for gold-answer review being ongoing work, not a one-time dataset-creation step.
Because the judge is itself an LLM call, re-running this cell may return different scores or a
different rationale even though nothing else about the question changed.

## 4. Example 2 — baseline vs. agentic, quantified

A cross-company question: comparing Apple and 3M requires pulling figures from *two* different
companies' data. Baseline does one vector search with one query embedding; the Module 6 agent
can call `get_financials` twice, once per company. Running both for real.


In [7]:
Q2 = next(q for q in QUESTIONS if q["id"] == "apple-3m-margin-2024")

chunks_2 = retrieve_chunks(embed_question(Q2["question"]), k=5)
baseline_context_2 = build_context(chunks_2)
baseline_answer_2 = generate_answer(Q2["question"], chunks_2)

print("===== baseline =====")
print(baseline_answer_2)


===== baseline =====
I can report 3M’s FY2024 figures from the provided context: net income attributable to 3M = $4,173 million and net sales = $24,575 million (FY2024). That implies a net income margin of about 4,173 / 24,575 ≈ 17.0%.

The context does not include Apple’s FY2024 net income or net sales, so I don’t know Apple’s figures and cannot determine which company had the higher net income margin.


In [8]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_6_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_6_TOOLS

agent = build_agent(MODULE_6_TOOLS, MODULE_6_STRATEGY_PROMPT)
agent_result_2 = agent.invoke(initial_state(Q2["question"]))
agent_answer_2 = agent_result_2["answer"]
agent_context_2 = agent_result_2["growing_knowledge"]

print("===== agentic (Module 6 toolset) =====")
print(agent_answer_2)


[strategy] iteration 1: 2 tool call(s) planned
    - get_financials({'company_id': 'APPLE'})
    - get_financials({'company_id': '3M'})
[tools] get_financials({'company_id': 'APPLE'}) -> 10 chunk(s)
[tools] get_financials({'company_id': '3M'}) -> 10 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end
===== agentic (Module 6 toolset) =====
Summary (fiscal year 2024):

- Apple: Total net sales (revenue) = $391,035,000,000; Net income = $93,736,000,000 (source: AAPL:2024-12-31). Net income margin = 93,736 / 391,035 ≈ 23.97% (calculated from AAPL:2024-12-31).

- 3M: Total net sales (revenue) = $24,575,000,000; Net income = $4,173,000,000 (source: MMM:2024-12-31). Net income margin = 4,173 / 24,575 ≈ 16.98% (calculated from MMM:2024-12-31).

Which had the higher net income margin: Apple had the higher net income margin (~24.0% vs ~17.0%) (sources: AAPL:2024-12-31; MMM:2024-12-31). 

If you want, I can show the exact division steps used to compute each margin.


In [9]:
from financial_advisor.evaluation.metrics import aggregate_by_system

result_2_baseline = judge_answer(Q2["question"], baseline_answer_2, Q2["ground_truth"], context=baseline_context_2)
result_2_baseline["system"] = "baseline"

result_2_agentic = judge_answer(Q2["question"], agent_answer_2, Q2["ground_truth"], context=agent_context_2)
result_2_agentic["system"] = "agentic"

for r in (result_2_baseline, result_2_agentic):
    print(r["system"], {k: v for k, v in r.items() if k not in ("rationale", "system")})

print()
print(aggregate_by_system([result_2_baseline, result_2_agentic]))


baseline {'correctness': 3, 'completeness': 3, 'grounding': 5}
agentic {'correctness': 5, 'completeness': 5, 'grounding': 5}

{'baseline': {'correctness': 3.0, 'completeness': 3.0, 'grounding': 5.0}, 'agentic': {'correctness': 5.0, 'completeness': 5.0, 'grounding': 5.0}}


**What happened, for example.** Baseline's single k=5 search surfaced 3M's figures but not
Apple's — and, honestly, said "I don't know" for the Apple half rather than guessing. That's
the right behavior for a system with no way to fetch it, but it's still an incomplete answer:
in one run, the judge scored `correctness`/`completeness` at 3, `grounding` still a clean 5
(nothing it said was invented). The agent called `get_financials` once per company, got both
figures with citations, and computed both margins correctly — `aggregate_by_system` puts a
number on the gap instead of "the agent seems better": in that same run, baseline averaged
3.0/3.0/5.0 across the three dimensions, the agent 5.0/5.0/5.0. This is the concrete version of
what this whole course has been arguing for since Module 2 — not "agentic feels more
sophisticated," but a measured score difference on a head-to-head question neither system saw
before. Exact numbers will vary run to run, since the judge is itself a non-deterministic LLM
call — but the qualitative gap, baseline missing a company it has no way to fetch vs. the agent
retrieving both, is structural, not a fluke of one sample.

## 5. Example 3 — a multi-part question, and why "wrong" isn't always the right verdict

A single question bundling three genuinely independent sub-questions: a figure, a comparative
claim, and a qualitative risk factor. The point of this section is to show what a **holistic**
score hides that a **decomposed** one recovers.


In [10]:
Q3 = next(q for q in QUESTIONS if q["id"] == "apple-2024-three-part")

chunks_3 = retrieve_chunks(embed_question(Q3["question"]), k=5)
context_3 = build_context(chunks_3)
real_answer_3 = generate_answer(Q3["question"], chunks_3)

print("Question:", Q3["question"])
print("\nBaseline answer:\n", real_answer_3)


Question: For fiscal year 2024, what was Apple total net sales, which product category had the fastest year-over-year growth, and what does Apple identify as a key risk related to its supply chain concentration?

Baseline answer:
 - Total net sales (fiscal 2024): $391,035 million.  
- Fastest year‑over‑year growth by category (2024 vs 2023): Services, up 13%.  
- Key risk related to supply chain concentration: I don't know — the provided context does not state what Apple identifies as a key risk related to its supply chain concentration.


In [11]:
from financial_advisor.evaluation.judge import judge_answer_decomposed

holistic_real = judge_answer(Q3["question"], real_answer_3, Q3["ground_truth"], context=context_3)
decomposed_real = judge_answer_decomposed(Q3["question"], real_answer_3, Q3["ground_truth"])

print("Holistic:", {k: v for k, v in holistic_real.items() if k != "rationale"})
print("\nDecomposed, correctness =", decomposed_real["correctness"])
for c in decomposed_real["sub_claims"]:
    print(f"  [{c['verdict']}] {c['claim']}")


Holistic: {'correctness': 3, 'completeness': 3, 'grounding': 5}

Decomposed, correctness = 3.0
  [partially_correct] Total net sales for fiscal year 2024 (and year‑over‑year change)
  [correct] Which product category had the fastest year‑over‑year growth and its growth rate
  [not_addressed] What key risk does Apple identify related to its supply chain concentration


**What happened, part 1 — a real, honest partial answer, for example.** The baseline got the
total net sales figure right (though it omitted the gold answer's "+2% YoY" framing, which in
one run led the judge to mark it `partially_correct` rather than a flat `correct`) and correctly
named Services as the fastest-growing category, but couldn't find Apple's supply-chain risk
language in its top-5 retrieved chunks — a single query embedding has to serve three very
different information needs (a headline number, a segment comparison, a risk-factors paragraph)
at once, and it lost the third one. In that run, the holistic judge already handled this
reasonably (`completeness=3`, not a flat failure) — but the decomposed view is strictly more
useful: it names *exactly* which one of three sub-claims is missing (`not_addressed`), confirms
one other is fully `correct`, and flags the net-sales sub-claim as only `partially_correct` — a
level of precision "completeness=3" alone doesn't give a reader trying to decide what to fix
next.

In [12]:
# A constructed variant, generated from the SAME retrieved context above — imagine the answer
# step, given identical retrieval, asserted a wrong claim instead of an honest "I don't know".
# This isolates one specific, common failure mode: a confidently wrong claim mixed with a
# correct one, rather than an honest gap — the case a holistic score handles worst.
flawed_answer_3 = (
    "For fiscal year 2024, Apple total net sales were $391.035 billion. iPhone was the "
    "fastest-growing product category year over year, driven by strong upgrade demand."
)

holistic_flawed = judge_answer(Q3["question"], flawed_answer_3, Q3["ground_truth"], context=context_3)
decomposed_flawed = judge_answer_decomposed(Q3["question"], flawed_answer_3, Q3["ground_truth"])

print("Holistic:", {k: v for k, v in holistic_flawed.items() if k != "rationale"})
print("\nDecomposed, correctness =", decomposed_flawed["correctness"])
for c in decomposed_flawed["sub_claims"]:
    print(f"  [{c['verdict']}] {c['claim']}")


Holistic: {'correctness': 1, 'completeness': 3, 'grounding': 1}

Decomposed, correctness = 2.33
  [correct] For fiscal year 2024, what were Apple's total net sales?
  [incorrect] Which product category had the fastest year-over-year growth in fiscal 2024?
  [not_addressed] What key risk does Apple identify related to its supply chain concentration?


**What happened, part 2 — the headline result, for example.** This answer nails the net-sales
figure, asserts iPhone (not Services) grew fastest — flatly contradicting the gold answer and
the retrieved context, which both say Services — and never addresses supply-chain risk at all.
In one run, the **holistic** judge's `correctness` collapsed to 1: "contradicts the gold answer
on its central claim." `grounding` collapsed to 1 too, for a reason worth noting — this isn't
just "doesn't match gold," the iPhone/upgrade-demand claim isn't supported by the retrieved
context either, which is a genuine hallucination, not merely an unlucky framing choice. That's
the halo effect in action on `correctness`: one confidently wrong, unsupported claim drags the
score for the whole answer down to "fails almost entirely," even though `completeness=3` (from
the same holistic call) showed the judge still recognized two of three parts were attempted.
The **decomposed** judge's `correctness` came out at 2.33/5 in that same run: `correct` on the
sales figure, `incorrect` on the growth category, `not_addressed` on the risk factor — a
genuinely different, more actionable verdict than "wrong." A team reading only the holistic
`correctness=1` would conclude this retrieval/generation path is broken across the board; the
decomposed score correctly narrows the problem to one specific, mislabeled sub-claim.
Re-running this cell may produce different exact scores, but the structural pattern — holistic
collapsing a partial failure to "broken," decomposed isolating it — is the point.

## 6. Aggregate metrics across everything run in this notebook

Individual examples make the mechanism visible; a real eval run needs the aggregate.
`aggregate_scores` and `aggregate_by_system` turn the results collected above into the kind of
number that belongs in a regression gate or a dashboard, not just a transcript.


In [13]:
from financial_advisor.evaluation.metrics import aggregate_scores

all_results = [result_1, result_2_baseline, result_2_agentic, holistic_real, holistic_flawed]
for r in all_results:
    r.setdefault("system", "baseline")

print("Overall (every judged answer in this notebook):")
print(aggregate_scores(all_results))

print("\nBy system:")
print(aggregate_by_system(all_results))


Overall (every judged answer in this notebook):
{'correctness': 3.0, 'completeness': 3.4, 'grounding': 4.2}

By system:
{'baseline': {'correctness': 2.5, 'completeness': 3.0, 'grounding': 4.0}, 'agentic': {'correctness': 5.0, 'completeness': 5.0, 'grounding': 5.0}}


Two answers here came from the agentic (Module 6) system, the rest from the baseline —
small enough that this table is illustrative, not a real benchmark result. A production eval run
would score every system against the full `data/eval/questions.json` set (and a much larger one
than three questions), store the per-question results, and diff `aggregate_by_system` output
across commits — exactly the shape a CI regression gate on retrieval-quality changes would take.


## 7. Production concerns: what running this for real, continuously, actually costs

This notebook ran the judge a handful of times, by hand, once. Operating it as a real evaluation
system raises different questions than getting `judge_answer` to return a sensible score:

**Cost.** Every judged answer costs at least one extra LLM call beyond whatever the system under
test cost — `judge_answer_decomposed` costs comparably more in output tokens per call (2-4
sub-claims, each with its own rationale). An eval run of `N` questions across `M` systems is
`N × M` judge calls, before counting the systems' own cost. Evaluation is not free just because
it happens after generation — budget for it the same way generation cost is budgeted for.

**Monitoring.** `judge_answer`'s `rationale` field isn't cosmetic — logging it alongside the
numeric scores (not just the scores) is what makes a bad score actionable rather than just a
number that dropped. A dashboard tracking `aggregate_by_system` output over time, per commit or
per deploy, turns "did the last prompt change help" from a question someone has to remember to
manually check into something that shows up automatically.

**Graph refresh.** `FinancialPeriod` data ages — Sharadar publishes new fundamentals every
quarter, and a gold answer built against last quarter's `HAS_FINANCIALS` figures silently goes
stale the moment new data lands (nothing currently invalidates or re-dates a gold answer when the
underlying `FinancialPeriod` it cites gets superseded by a newer one for the same company). A
real deployment needs either gold answers pinned to a specific `calendardate` (already true here
— every gold answer in `data/eval/questions.json` names the fiscal year explicitly) or a process
that reviews the eval dataset each time the graph refreshes.

**Governance.** Section 3's example — the judge under-crediting an answer because the gold
answer's own framing was ambiguous — is the concrete argument for why an eval dataset needs an
owner and a review process, not just an author. `source` fields exist in
`data/eval/questions.json` specifically so a reviewer doesn't have to trust this notebook's
authorship; they can independently re-verify each gold answer against `FinancialPeriod` or the
underlying filing. Any prompt, model, or retrieval change that's expected to move scores should
re-run against the full dataset before shipping — the same discipline as a unit-test suite, run
against a judge instead of an assertion.

## What this module actually demonstrated

- A judge built to resist known bias patterns (reference-based only, anchored rubric,
  rationale-first fields, separated grounding, decomposition) — not by assumption, but shown
  against real answers from real systems in this project.
- Two different kinds of gold answer (vendor-verified `FinancialPeriod` figures vs. curated
  narrative facts) — and a worked example of the judge inheriting an ambiguity from the
  *narrative* half, which no amount of judge engineering alone fixes.
- A quantified, not just anecdotal, comparison between the Module 1 baseline and the Module 6
  agentic system on the same question — in one run, 3.0/3.0/5.0 vs. 5.0/5.0/5.0 across
  correctness/completeness/grounding, not just "the agent seems better." (Exact numbers vary
  run to run, since the judge is itself an LLM call; the qualitative gap is the point.)
- The concrete value of decomposition: in one run, the same flawed answer scored
  `correctness=1` holistically and `correctness=2.33` decomposed — the difference between "this
  system is broken" and "this system got one specific thing wrong," which is the difference
  between a debugging dead end and a debugging lead.

Rigorous evaluation isn't a nice-to-have bolted onto the end of a RAG project — it's what turns
"the agentic loop feels better" into a number a team can actually make a shipping decision on.